# Independence in Privacy Accounting

**Independence** means a user's privacy cost is charged only in proportion to their
personal contribution to each query result.  This notebook demonstrates the benefit
of independence both in TCT threshold queries and in interleaved non-TCT queries.

## Accountants compared

| Accountant | TCT rounds | Non-TCT rounds |
|---|---|---|
| Standard DP (non-independent) | ε every round | ε every round |
| Standard DP (independent) | ε only if contributed | ε only if participated |
| Adaptive TCT (independent) | K charge only if contributed to top | ε only if participated |

## Non-TCT query model

Every `INTERLEAVE_EVERY` TCT rounds a non-TCT query fires.  Each user
participates in it independently with probability `P_PARTICIPATE`.  Under
independent accounting only participating users pay `ε_nontct`; under
non-independent accounting all active users pay regardless.

**Sections:**
1. Parameters and setup
2. Single run — retention and budget curves
3. Effect of participation probability on the independent advantage
4. Per-user cost stratification by contribution count

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

from simulation.query import compute_round_conversions, laplace_threshold_query
from simulation.accountants.adaptive_tct import AdaptiveTCTAccountant
from simulation.accountants.standard_dp import StandardDPAccountant
from simulation.calibration import q_from_epsilon, set_threshold_for_hit_rate
from simulation.datasets import generate_synthetic_population, PopulationConfig

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## Section 1: Parameters

In [ ]:
# ── Privacy parameters ────────────────────────────────────────────────────────
EPSILON          = 0.1
DELTA            = 1e-3

# ── Non-TCT interleaving ──────────────────────────────────────────────────────
INTERLEAVE_EVERY = 10       # one non-TCT query every N TCT rounds
EPSILON_NONTCT   = EPSILON
P_PARTICIPATE    = 0.3      # probability each user participates in a non-TCT query

# ── Population / simulation parameters ───────────────────────────────────────
N_USERS         = 300
N_ROUNDS        = 600
EPS_BUDGET      = 20.0
DELTA_BUDGET    = 1.0
TARGET_HIT_RATE = 0.05
POP_ALPHA       = 0.3
POP_BETA        = 3.0
SEED            = 42

q = q_from_epsilon(EPSILON)

config = PopulationConfig(
    n_users=N_USERS, alpha=POP_ALPHA, beta=POP_BETA,
    epsilon_budget=EPS_BUDGET, delta_budget=DELTA_BUDGET,
)
calib_pop = generate_synthetic_population(config, rng=np.random.default_rng(SEED))
THRESHOLD, actual_hr = set_threshold_for_hit_rate(
    calib_pop, epsilon=EPSILON, target_hit_rate=TARGET_HIT_RATE,
    n_pilot=2000, rng=np.random.default_rng(SEED + 9999),
)

print(f'ε={EPSILON}, δ={DELTA}, q={q:.4f}')
print(f'Threshold T={THRESHOLD:.3f},  empirical hit rate={actual_hr:.3f}')
print(f'Non-TCT query every {INTERLEAVE_EVERY} rounds, ε={EPSILON_NONTCT}, p_participate={P_PARTICIPATE}')
print()
acc_adap = AdaptiveTCTAccountant(EPSILON, DELTA, independent=True)
print('Per-round expected cost (independent, p_participate):')
print(f'  Adaptive TCT    TCT rounds: ~{actual_hr * acc_adap.compositions(1) * EPSILON:.4f}ε  (hits are rare)')
print(f'  Standard DP ind TCT rounds: ~{actual_hr * EPSILON:.4f}ε  (only contributors pay)')
print(f'  Standard DP     TCT rounds: {EPSILON:.4f}ε  (everyone pays)')
print(f'  Non-TCT rounds (independent): ~{P_PARTICIPATE * EPSILON_NONTCT:.4f}ε  (p={P_PARTICIPATE})')
print(f'  Non-TCT rounds (non-indep):   {EPSILON_NONTCT:.4f}ε  (everyone pays)')

## Section 2: Single Run — Retention and Budget Curves

In [ ]:
def run_interleaved(accountant, interleave_every, eps_nontct,
                    p_participate, independent_nontct, seed=SEED):
    """
    Simulate TCT rounds with interleaved non-TCT queries.

    Non-TCT query model:
      - Fires every `interleave_every` TCT rounds (0 = never).
      - Each active user participates independently with probability `p_participate`.
      - independent_nontct=True : only participating users pay eps_nontct.
      - independent_nontct=False: all active users pay eps_nontct regardless.

    Returns:
        active_frac    list[float]  active fraction after each round
        avg_eps        list[float]  mean ε remaining among active users
        contrib_counts dict[uid -> int]  rounds where user contributed (TCT)
        final_eps      dict[uid -> float] final ε remaining per user
    """
    rng = np.random.default_rng(seed)
    pop = generate_synthetic_population(config, rng=np.random.default_rng(seed))
    contrib_counts = defaultdict(int)

    active_frac, avg_eps = [], []

    for r in range(N_ROUNDS):
        active = pop.active_users
        if not active:
            break

        # ── TCT round ─────────────────────────────────────────────────────────
        true_count, contributions = compute_round_conversions(active, rng)
        is_top, _ = laplace_threshold_query(true_count, THRESHOLD, EPSILON, rng)

        for i, user in enumerate(active):
            if contributions[i]:
                contrib_counts[user.user_id] += 1
            e, d = accountant.charge(
                user=user, is_top=is_top,
                contributed=contributions[i], round_num=r,
            )
            user.epsilon_spent += e
            user.delta_spent   += d

        # ── Non-TCT query ──────────────────────────────────────────────────────
        if interleave_every > 0 and (r + 1) % interleave_every == 0:
            for user in pop.active_users:
                participates = rng.random() < p_participate
                if not independent_nontct or participates:
                    user.epsilon_spent += eps_nontct

        pop.drop_depleted()
        eps_dist = pop.epsilon_remaining_distribution()
        active_frac.append(pop.active_fraction)
        avg_eps.append(float(np.mean(eps_dist)) if eps_dist else 0.0)

    final_eps = {u.user_id: max(u.epsilon_remaining, 0.0) for u in pop.all_users}
    return active_frac, avg_eps, dict(contrib_counts), final_eps


def half_life(active_frac):
    for r, f in enumerate(active_frac):
        if f <= 0.5:
            return r
    return len(active_frac)


print('Running simulations...')
# Adaptive TCT (independent) + independent non-TCT
tct_act, tct_eps, tct_contrib, tct_final = run_interleaved(
    AdaptiveTCTAccountant(EPSILON, DELTA, independent=True),
    INTERLEAVE_EVERY, EPSILON_NONTCT, P_PARTICIPATE, independent_nontct=True)

# Standard DP (independent) + independent non-TCT
dp_ind_act, dp_ind_eps, dp_ind_contrib, dp_ind_final = run_interleaved(
    StandardDPAccountant(EPSILON, independent=True),
    INTERLEAVE_EVERY, EPSILON_NONTCT, P_PARTICIPATE, independent_nontct=True)

# Standard DP (non-independent) + non-independent non-TCT (baseline)
dp_act, dp_eps, dp_contrib, dp_final = run_interleaved(
    StandardDPAccountant(EPSILON, independent=False),
    INTERLEAVE_EVERY, EPSILON_NONTCT, P_PARTICIPATE, independent_nontct=False)

print('Done.')

In [ ]:
def pad(curve, n):
    return list(curve) + [curve[-1]] * (n - len(curve))

n = max(len(tct_act), len(dp_ind_act), len(dp_act)) + 1
rnds = list(range(n))

hl_tct    = half_life(tct_act)
hl_dp_ind = half_life(dp_ind_act)
hl_dp     = half_life(dp_act)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(rnds, pad(tct_act,    n), color='steelblue',  lw=2,
        label=f'Adaptive TCT  (independent,  hl={hl_tct})')
ax.plot(rnds, pad(dp_ind_act, n), color='seagreen',   lw=2,
        label=f'Standard DP   (independent,  hl={hl_dp_ind})')
ax.plot(rnds, pad(dp_act,     n), color='gray',       lw=2, linestyle='--',
        label=f'Standard DP   (non-independent, hl={hl_dp})')
ax.axhline(0.5, color='silver', lw=1, linestyle=':')
ax.set_xlabel('Round')
ax.set_ylabel('Fraction of users active')
ax.set_title('User Retention')
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(rnds, pad(tct_eps,    n), color='steelblue',  lw=2, label='Adaptive TCT (independent)')
ax.plot(rnds, pad(dp_ind_eps, n), color='seagreen',   lw=2, label='Standard DP (independent)')
ax.plot(rnds, pad(dp_eps,     n), color='gray',       lw=2, linestyle='--', label='Standard DP (non-independent)')
ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('Round')
ax.set_ylabel('Avg ε remaining (active users)')
ax.set_title('Average Budget Remaining')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle(
    f'N={N_USERS}, ε={EPSILON}, δ={DELTA}  |  '
    f'non-TCT every {INTERLEAVE_EVERY} rounds, p_participate={P_PARTICIPATE}',
    fontsize=10
)
plt.tight_layout()
plt.show()

print(f'Half-life: Adaptive TCT (ind)={hl_tct},  Standard DP (ind)={hl_dp_ind},  Standard DP={hl_dp}')
print(f'TCT vs DP (ind):     {hl_tct - hl_dp_ind:+d} rounds')
print(f'DP (ind) vs DP:      {hl_dp_ind - hl_dp:+d} rounds')

## Section 3: Effect of Participation Probability

How does the independence advantage change as non-TCT participation becomes more or less likely?

- **p=1.0**: every user participates in every non-TCT query — independent and non-independent
  non-TCT costs are identical; advantage comes only from TCT independence.
- **p→0**: non-TCT cost vanishes for independent accountants; TCT independence
  dominates the comparison.

In [ ]:
p_sweep = [0.1, 0.2, 0.3, 0.5, 0.7, 1.0]

hl_tct_sweep    = []
hl_dp_ind_sweep = []
hl_dp_sweep     = []

for p in p_sweep:
    a, *_ = run_interleaved(AdaptiveTCTAccountant(EPSILON, DELTA, independent=True),
                            INTERLEAVE_EVERY, EPSILON_NONTCT, p, independent_nontct=True)
    b, *_ = run_interleaved(StandardDPAccountant(EPSILON, independent=True),
                            INTERLEAVE_EVERY, EPSILON_NONTCT, p, independent_nontct=True)
    c, *_ = run_interleaved(StandardDPAccountant(EPSILON, independent=False),
                            INTERLEAVE_EVERY, EPSILON_NONTCT, p, independent_nontct=False)
    hl_tct_sweep.append(half_life(a))
    hl_dp_ind_sweep.append(half_life(b))
    hl_dp_sweep.append(half_life(c))

gap_tct_dp    = [hl_tct_sweep[i]    - hl_dp_sweep[i] for i in range(len(p_sweep))]
gap_dp_ind_dp = [hl_dp_ind_sweep[i] - hl_dp_sweep[i] for i in range(len(p_sweep))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(p_sweep, hl_tct_sweep,    'o-', color='steelblue', lw=2, label='Adaptive TCT (independent)')
ax.plot(p_sweep, hl_dp_ind_sweep, 's-', color='seagreen',  lw=2, label='Standard DP (independent)')
ax.plot(p_sweep, hl_dp_sweep,     '^--',color='gray',      lw=2, label='Standard DP (non-independent)')
ax.set_xlabel('p_participate (non-TCT query)')
ax.set_ylabel('Half-life (rounds)')
ax.set_title('Half-Life vs Participation Probability')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(p_sweep, gap_tct_dp,    'o-', color='steelblue', lw=2, label='Adaptive TCT − Standard DP')
ax.plot(p_sweep, gap_dp_ind_dp, 's-', color='seagreen',  lw=2, label='Standard DP (ind) − Standard DP')
ax.axhline(0, color='gray', lw=0.8, linestyle='--')
ax.set_xlabel('p_participate (non-TCT query)')
ax.set_ylabel('Half-life advantage over non-independent DP (rounds)')
ax.set_title('Independence Advantage vs Participation Probability')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle(f'ε={EPSILON}, δ={DELTA}, non-TCT every {INTERLEAVE_EVERY} rounds', fontsize=11)
plt.tight_layout()
plt.show()

header = '{:>6}  {:>8}  {:>10}  {:>8}  {:>12}  {:>14}'.format(
    'p', 'TCT HL', 'DP ind HL', 'DP HL', 'TCT−DP gap', 'DP ind−DP gap')
print(header)
print('-' * 66)
for i, p in enumerate(p_sweep):
    print(f'{p:>6.1f}  {hl_tct_sweep[i]:>8}  {hl_dp_ind_sweep[i]:>10}  '
          f'{hl_dp_sweep[i]:>8}  {gap_tct_dp[i]:>+12}  {gap_dp_ind_dp[i]:>+14}')

## Section 4: Per-User Cost Stratification by Contribution Count

Under independent accounting, privacy cost should concentrate on users who
frequently contributed to queries.  Users who rarely contributed should retain
most of their budget.

Contribution count here is the number of TCT rounds in which the user personally
generated a conversion (independent of whether the round was a top).

In [ ]:
# Bucket users by contribution count (TCT rounds where contributed=True)
def stratify_by_contrib(contrib_counts, final_eps):
    buckets = defaultdict(list)
    for uid, eps in final_eps.items():
        count = contrib_counts.get(uid, 0)
        # Buckets: 0, 1-5, 6-20, 21+
        if count == 0:       bucket = '0'
        elif count <= 5:     bucket = '1-5'
        elif count <= 20:    bucket = '6-20'
        else:                bucket = '21+'
        buckets[bucket].append(eps)
    return buckets

tct_buckets    = stratify_by_contrib(tct_contrib,    tct_final)
dp_ind_buckets = stratify_by_contrib(dp_ind_contrib, dp_ind_final)
dp_buckets     = stratify_by_contrib(dp_contrib,     dp_final)

bucket_labels = ['0', '1-5', '6-20', '21+']

def bucket_means(buckets):
    return [np.mean(buckets[k]) if buckets[k] else 0.0 for k in bucket_labels]

def bucket_ns(buckets):
    return [len(buckets[k]) for k in bucket_labels]

tct_means    = bucket_means(tct_buckets)
dp_ind_means = bucket_means(dp_ind_buckets)
dp_means     = bucket_means(dp_buckets)
ns           = bucket_ns(tct_buckets)

x_pos = np.arange(len(bucket_labels))
width = 0.25

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.bar(x_pos - width,   tct_means,    width, color='steelblue', alpha=0.85, label='Adaptive TCT (ind)')
ax.bar(x_pos,           dp_ind_means, width, color='seagreen',  alpha=0.85, label='Standard DP (ind)')
ax.bar(x_pos + width,   dp_means,     width, color='gray',      alpha=0.85, label='Standard DP')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'{lbl}\n(n={ns[i]})' for i, lbl in enumerate(bucket_labels)])
ax.set_xlabel('TCT contribution count')
ax.set_ylabel('Mean ε remaining at end of simulation')
ax.set_title('Budget Remaining by Contribution Bucket')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(0, color='black', lw=0.5)

ax = axes[1]
adv_tct    = [tct_means[i]    - dp_means[i] for i in range(len(bucket_labels))]
adv_dp_ind = [dp_ind_means[i] - dp_means[i] for i in range(len(bucket_labels))]
ax.plot(bucket_labels, adv_tct,    'o-', color='steelblue', lw=2, label='Adaptive TCT (ind) − Standard DP')
ax.plot(bucket_labels, adv_dp_ind, 's-', color='seagreen',  lw=2, label='Standard DP (ind) − Standard DP')
ax.axhline(0, color='gray', lw=0.8, linestyle='--')
ax.set_xlabel('TCT contribution count')
ax.set_ylabel('Δ ε remaining vs non-independent DP')
ax.set_title('Independence Advantage by Contribution Bucket')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle(
    f'N={N_USERS}, ε={EPSILON}, δ={DELTA}  |  '
    f'non-TCT every {INTERLEAVE_EVERY} rounds, p={P_PARTICIPATE}  |  budget={EPS_BUDGET}',
    fontsize=10
)
plt.tight_layout()
plt.show()

print(f'\n{"Bucket":<8}  {"n":>5}  {"TCT ind":>9}  {"DP ind":>9}  {"DP":>9}  {"TCT adv":>9}  {"DP ind adv":>10}')
print('-' * 66)
for i, lbl in enumerate(bucket_labels):
    print(f'{lbl:<8}  {ns[i]:>5}  {tct_means[i]:>9.3f}  {dp_ind_means[i]:>9.3f}  '
          f'{dp_means[i]:>9.3f}  {adv_tct[i]:>+9.3f}  {adv_dp_ind[i]:>+10.3f}')